In [ ]:
#| include: false
import pandas as pd
import numpy as np
import plotly.io as pio
from IPython.display import display, Markdown

pio.renderers.default = 'notebook'
pio.templates.default = 'plotly_white'

from aavolve.report_helpers import *

# parameters (overridden by papermill)
group_id = 'GROUP'
manifest = 'out/qc/group_reports/GROUP_manifest.tsv'


<script type="text/javascript">
function aavolveResizePlotly(container) {
  if (typeof require === "undefined") return;
  require(["plotly"], function(Plotly) {
    (container || document).querySelectorAll(".plotly-graph-div").forEach(function(gd) {
      try { Plotly.Plots.resize(gd); } catch (e) {}
    });
  });
}
document.addEventListener("shown.bs.tab", function(e) {
  var selector = e.target && e.target.getAttribute && e.target.getAttribute("data-bs-target");
  var pane = selector ? document.querySelector(selector) : null;
  aavolveResizePlotly(pane);
});
window.addEventListener("resize", function() { aavolveResizePlotly(); });
</script>


In [ ]:
#| include: false
manifest_df = pd.read_csv(manifest, sep='	')
samples = manifest_df['sample'].tolist()
parent_file = manifest_df['parent_file'].iloc[0]
reference_file = manifest_df['reference_file'].iloc[0]
color_map = parent_colors(manifest_df['parent_frequencies'].iloc[0])


In [ ]:
display(Markdown('This report aggregates all samples which share the same parent/reference combination.'))
display(Markdown(f'- Group ID: `{group_id}`'))
display(Markdown(f'- Parent file: `{parent_file}`'))
display(Markdown(f'- Reference file: `{reference_file}`'))
display(Markdown(f"- Samples: `{', '.join(samples)}`"))


In [ ]:
#| echo: false
parents_warn = manifest_df["parents_dropped_warning"].iloc[0] if "parents_dropped_warning" in manifest_df.columns else ""
variant_warn = manifest_df["variant_window_warning"].iloc[0] if "variant_window_warning" in manifest_df.columns else ""
display_warning_file(parents_warn, title="Warning")
display_warning_file(variant_warn, title="Warning")


In [ ]:
#| title: Read counts at each processing stage (all samples)

dfs = []
for row in manifest_df.itertuples(index=False):
    df = import_read_count_data(row.read_counts, row.seq_tech)
    df = df[df['File type'] != 'Distinct at nucleotide level']
    df = df[df['File type'] != 'Distinct at amino acid level']
    df['sample'] = row.sample
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)

fig = px.line(df_all, x='File type', y='Count', color='sample', markers=True)
fig.update_xaxes(tickangle=90, automargin=True)
fig.update_layout(
    margin=dict(l=60, r=40, t=60, b=260),
    legend=dict(
        title_text='Sample',
        orientation='h',
        entrywidth=140,
        entrywidthmode='pixels',
        x=0,
        xanchor='left',
        y=-0.45,
        yanchor='top',
        font=dict(size=10),
    ),
)
fig

In [ ]:
#| title: Fraction of reads retained after each filter (all samples)

fig = px.line(df_all, x='File type', y='Fraction of reads', color='sample', markers=True)
fig.update_xaxes(tickangle=90, automargin=True)
fig.update_layout(
    margin=dict(l=60, r=40, t=60, b=260),
    legend=dict(
        title_text='Sample',
        orientation='h',
        entrywidth=140,
        entrywidthmode='pixels',
        x=0,
        xanchor='left',
        y=-0.45,
        yanchor='top',
        font=dict(size=10),
    ),
)
fig

In [ ]:
#| title: QC summary table

from IPython.display import display
rows = []
for row in manifest_df.itertuples(index=False):
    # reads passing all filters (count + percent of input)
    try:
        n_pass, frac_pass = reads_passing_all_filters(row.read_counts, row.seq_tech)
    except Exception:
        n_pass, frac_pass = 0, 0.0
    # coverage depth summary
    cov_min = cov_median = cov_low = None
    if hasattr(row, "coverage_depth") and isinstance(row.coverage_depth, str) and row.coverage_depth:
        try:
            cov = coverage_depth_summary(row.coverage_depth)
            cov_min = cov["min"]
            cov_median = cov["median"]
            cov_low = cov["low_frac"]
        except Exception:
            pass
    rows.append({
        "sample": row.sample,
        "reads_passing_all": n_pass,
        "pct_passing_all": frac_pass*100,
        "coverage_min": cov_min,
        "coverage_median": cov_median,
        "coverage_low_%(<10)": None if cov_low is None else cov_low*100,
        "trim": getattr(row, "trim", None),
        "include_non_parental": getattr(row, "include_non_parental", None),
    })
display(pd.DataFrame(rows))


In [ ]:
#| title: Trimming overhang summary (pre/post)

from IPython.display import display
rows = []
for row in manifest_df.itertuples(index=False):
    pre_frac = post_frac = None
    try:
        df_pre, _ = msa_overhangs(row.pretrim_msa)
        pre_frac = float(((df_pre.overhang_5 > 5) | (df_pre.overhang_3 > 5)).mean())
    except Exception:
        pass
    post_path = getattr(row, "posttrim_msa", "")
    if isinstance(post_path, str) and post_path:
        try:
            df_post, _ = msa_overhangs(post_path)
            post_frac = float(((df_post.overhang_5 > 5) | (df_post.overhang_3 > 5)).mean())
        except Exception:
            pass
    rows.append({"sample": row.sample, "pre_overhang_%(>5bp)": None if pre_frac is None else pre_frac*100, "post_overhang_%(>5bp)": None if post_frac is None else post_frac*100})
display(pd.DataFrame(rows))


In [ ]:
#| title: Overall parent assignment summary (mean across variants)

summary_rows = []
for row in manifest_df.itertuples(index=False):
    df = pd.read_csv(row.parent_frequencies, sep='	')
    df = df[~df['parent'].astype(str).str.match('non_parental_\\d+')]
    df['parent'] = df['parent'].astype(str).str.replace('non_parental_\d+', 'non parental', regex=True)
    df_sum = df.groupby('parent', as_index=False)['frequency'].mean()
    df_sum['frequency'] = df_sum['frequency'] * 100
    df_sum['sample'] = row.sample
    summary_rows.append(df_sum)
summary_df = pd.concat(summary_rows, ignore_index=True)

fig = px.bar(
    summary_df,
    x='parent',
    y='frequency',
    color='parent',
    facet_col='sample',
    facet_col_wrap=2,
    labels={'frequency': 'Mean parent frequency (%)', 'parent': 'Parent'},
    color_discrete_map=color_map,
)
fig.update_xaxes(tickangle=90, automargin=True)
fig.update_layout(margin=dict(l=60, r=40, t=60, b=160), showlegend=False)
fig

In [ ]:
#| title: Non-parental variants (group summary)

from IPython.display import display, Markdown
rows = []
total_reads = 0
for row in manifest_df.itertuples(index=False):
    include = str(getattr(row, "include_non_parental", "False")) == "True"
    if not include:
        continue
    try:
        df = pd.read_csv(row.variant_freq_all, sep="	")
    except Exception:
        continue
    try:
        refcov = get_read_count(row.read_counts, row.seq_tech, "Filtered by reference coverage")
    except Exception:
        refcov = 0
    total_reads += refcov
    thr = float(getattr(row, "non_parental_freq", 0.0) or 0.0)
    df = df[(df.query_name == "non_parental") & (df.freq >= thr)].copy()
    if len(df) == 0:
        continue
    df["sample"] = row.sample
    df["read_count"] = (df["freq"].astype(float) * refcov).round().astype(int)
    rows.append(df)

if not rows:
    display(Markdown('No non-parental variants included in this group.'))
else:
    all_df = pd.concat(rows, ignore_index=True)
    def fmt_variant(r):
        pos=str(r.pos); ref=str(r.ref_bases); alt=str(r.query_bases)
        if ref==".": return f"{pos}ins{alt}"
        if alt==".": return f"{ref}{pos}del"
        return f"{ref}{pos}{alt}"
    all_df["variant"] = all_df.apply(fmt_variant, axis=1)
    agg = all_df.groupby("variant", as_index=False).agg(
        n_samples=("sample", "nunique"),
        samples=("sample", lambda xs: ", ".join(sorted(set(xs)))),
        total_read_count=("read_count", "sum"),
    )
    if total_reads > 0:
        agg["total_read_fraction"] = agg["total_read_count"] / total_reads
    else:
        agg["total_read_fraction"] = 0.0
    agg = agg.sort_values(["total_read_fraction", "total_read_count"], ascending=[False, False])
    display(agg)


In [ ]:
#| title: Assigned parents for top reads (per sample)
#| fig-width: 100%

for row in manifest_df.itertuples(index=False):
    display(Markdown(f'## {row.sample}'))
    fig = parent_heatmap(row.assigned_parents, row.parent_frequencies)
    if fig is not None:
        fig.show()

In [ ]:
#| title: Mean pairwise distance between top capsids

metric_rows = []
for row in manifest_df.itertuples(index=False):
    for label, path in [
        ('nt-first', row.dmat_nt_first),
        ('aa-first', row.dmat_aa_first),
    ]:
        dmat = np.loadtxt(path, ndmin=2)
        mask = ~np.eye(dmat.shape[0], dtype=bool)
        mean_distance = 0.0 if mask.sum() == 0 else float(dmat[mask].mean())
        metric_rows.append({'sample': row.sample, 'matrix': label, 'mean_distance': mean_distance})
metrics_df = pd.DataFrame(metric_rows)

fig = px.bar(metrics_df, x='sample', y='mean_distance', color='matrix', barmode='group',
             labels={'mean_distance': 'Mean pairwise distance', 'sample': 'Sample', 'matrix': 'Matrix'})
fig.update_layout(margin=dict(l=60, r=40, t=60, b=80))
fig

In [ ]:
#| title: Settings (per sample)

cols = [c for c in [
    "sample", "seq_tech", "trim", "adapter_5", "adapter_3", "anchors",
    "require_end_to_end_alignment", "include_non_parental", "non_parental_freq",
    "group_vars", "group_vars_dist", "max_group_distance", "minimap2_params"
] if c in manifest_df.columns]
display(manifest_df[cols])
